[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_11_12/02_tag_11_12_ar_vorhersage.ipynb)

# Tag 11 & 12 – 02: Einfache AR-Vorhersage als Ausgangspunkt

Ein autoregressives Modell (AR) nutzt ausschließlich **vergangene Werte der eigenen Zeitreihe**, um den nächsten Wert zu schätzen. In diesem Beispiel sind die Gewichte bewusst vorgegeben: Es findet noch kein Training statt. So wird transparent, wie eine Ein-Schritt-Prognose entsteht.

## Die AR-Idee

Bei einer Ordnung $p$ verwendet ein AR-Modell die letzten $p$ Werte:

$$\hat{x}_t = w_1 x_{t-p} + w_2 x_{t-p+1} + \dots + w_p x_{t-1}$$

Für $p=3$ bedeutet das: Die drei Werte direkt vor dem Zeitpunkt $t$ werden gewichtet addiert. Der letzte Wert erhält in diesem Beispiel das größte Gewicht, weil er oft die aktuellste Information enthält. Bei einem echten AR-Modell würden die Gewichte aus Trainingsdaten gelernt.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets

plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 7

## Eine kurze Zeitreihe und eine feste AR-Regel

Die Kurve ist absichtlich nicht perfekt autoregressiv: Sie enthält einen Verlauf und Messrauschen. Dadurch sieht man beim Vergleich sowohl brauchbare Schätzungen als auch Abweichungen. Die folgende Funktion berechnet für jeden möglichen Zeitpunkt eine Prognose aus den unmittelbar vorhergehenden `p` Werten.

In [ ]:
def erzeuge_zeitreihe(anzahl_werte=80, rauschen=0.10, seed=RANDOM_STATE):
    """Erzeugt eine kurze, periodische Messreihe."""
    rng = np.random.default_rng(seed)
    zeit = np.linspace(0, 5 * np.pi, anzahl_werte)
    werte = np.sin(zeit) + 0.20 * np.sin(2.3 * zeit) + rng.normal(0, rauschen, anzahl_werte)
    return werte


def ar_gewichte(ordnung_p, gesamtgewicht=0.95):
    """Feste, abnehmende Gewichte: der neueste Wert zählt am stärksten."""
    rohgewichte = 0.58 ** np.arange(ordnung_p - 1, -1, -1)
    return gesamtgewicht * rohgewichte / rohgewichte.sum()


def ar_ein_schritt_prognosen(reihe, ordnung_p, gewichte):
    """Liefert eine Prognose für jeden Zeitpunkt, an dem p Vergangenheitswerte vorliegen."""
    prognosen = np.full(len(reihe), np.nan)
    for zeitpunkt in range(ordnung_p, len(reihe)):
        vergangenheit = reihe[zeitpunkt - ordnung_p : zeitpunkt]
        prognosen[zeitpunkt] = np.dot(gewichte, vergangenheit)
    return prognosen


werte = erzeuge_zeitreihe()
p = 3
gewichte = ar_gewichte(p)
prognosen = ar_ein_schritt_prognosen(werte, p, gewichte)

print(f'AR({p}) mit Gewichten für [x(t-{p}), ..., x(t-1)]: {np.round(gewichte, 3)}')
print(f'Es entstehen {np.sum(~np.isnan(prognosen))} Ein-Schritt-Prognosen.')
print(f'Beispiel bei t={p}: Vergangenheit {np.round(werte[:p], 2)} → Prognose {prognosen[p]:.2f}; Istwert {werte[p]:.2f}')

## Interaktive Ein-Schritt-Prognose

Blau markiert sind die letzten $p$ Werte. Ihre gewichtete Summe ergibt den orangefarbenen Prognosepunkt. Der grüne Punkt zeigt den tatsächlich beobachteten nächsten Wert. Das gestrichelte Segment macht den Prognosefehler sichtbar.

Die Gewichte sind immer fest vorgegeben und summieren sich auf das eingestellte Gesamtgewicht. Deshalb kann die Regel unmittelbar erklärt werden, ohne dass ein Trainingsprozess im Hintergrund stattfindet.

In [ ]:
def zeige_ar_vorhersage(ordnung_p=3, gesamtgewicht=0.95, rauschen=0.10, zeitpunkt=20):
    werte = erzeuge_zeitreihe(rauschen=rauschen)
    gewichte = ar_gewichte(ordnung_p, gesamtgewicht)
    prognosen = ar_ein_schritt_prognosen(werte, ordnung_p, gewichte)
    t = max(ordnung_p, min(zeitpunkt, len(werte) - 1))
    vergangene_indizes = np.arange(t - ordnung_p, t)
    vergangenheit = werte[t - ordnung_p : t]
    prognose, istwert = prognosen[t], werte[t]
    mae = np.nanmean(np.abs(prognosen[ordnung_p:] - werte[ordnung_p:]))

    fig, (ax_gesamt, ax_detail) = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={'height_ratios': [2, 1]})
    x = np.arange(len(werte))
    ax_gesamt.plot(x, werte, 'o-', color='0.55', markersize=3.5, label='tatsächliche Zeitreihe')
    ax_gesamt.plot(x[ordnung_p:], prognosen[ordnung_p:], 'o', color='tab:orange', markersize=4, alpha=0.75, label='AR-Ein-Schritt-Prognosen')
    ax_gesamt.plot(vergangene_indizes, vergangenheit, 'o-', color='tab:blue', linewidth=2.8, markersize=7, label=f'letzte p={ordnung_p} Werte')
    ax_gesamt.scatter(t, prognose, color='tab:orange', s=100, zorder=4, label='ausgewählte Prognose')
    ax_gesamt.scatter(t, istwert, color='tab:green', s=100, zorder=5, label='tatsächlicher nächster Wert')
    ax_gesamt.vlines(t, min(prognose, istwert), max(prognose, istwert), color='crimson', linestyle='--', linewidth=2, label='absoluter Fehler')
    ax_gesamt.set(title=f'AR({ordnung_p}): Prognose bei t={t}', xlabel='Zeitindex', ylabel='Messwert')
    ax_gesamt.legend(ncol=3, loc='lower left')

    positionen = np.arange(1, ordnung_p + 1)
    beitraege = gewichte * vergangenheit
    ax_detail.bar(positionen, beitraege, color='tab:blue', alpha=0.80)
    ax_detail.axhline(0, color='0.3', linewidth=0.8)
    ax_detail.set_xticks(positionen, [f'x(t-{ordnung_p - i})' for i in range(ordnung_p)])
    ax_detail.set(title='Beiträge der vergangenen Werte zur Prognose', xlabel='vergangener Wert', ylabel='Gewicht × Wert')
    for position, beitrag, gewicht in zip(positionen, beitraege, gewichte):
        ax_detail.text(position, beitrag, f'w={gewicht:.2f}', ha='center', va='bottom' if beitrag >= 0 else 'top')
    plt.tight_layout()
    plt.show()

    print(f'AR({ordnung_p}): ŷ(t) = ' + ' + '.join(f'{w:.2f} · x(t-{ordnung_p - i})' for i, w in enumerate(gewichte)))
    print(f'Vergangene Werte: {np.round(vergangenheit, 2)}')
    print(f'Prognose ŷ({t}) = {prognose:.3f}; Istwert x({t}) = {istwert:.3f}; absoluter Fehler = {abs(prognose - istwert):.3f}')
    print(f'Mittlerer absoluter Fehler über alle möglichen Ein-Schritt-Prognosen: {mae:.3f}')
    if zeitpunkt != t:
        print(f'Der gewählte Zeitpunkt wurde auf den gültigen Bereich angepasst: t={t}.')


widgets.interact(
    zeige_ar_vorhersage,
    ordnung_p=widgets.IntSlider(value=3, min=1, max=6, step=1, description='Ordnung p'),
    gesamtgewicht=widgets.FloatSlider(value=0.95, min=0.50, max=1.20, step=0.05, description='Gesamtgewicht'),
    rauschen=widgets.FloatSlider(value=0.10, min=0.0, max=0.40, step=0.02, description='Rauschen'),
    zeitpunkt=widgets.IntSlider(value=20, min=1, max=79, step=1, description='Zeitpunkt t'),
);

## Kerngedanke

Die Ordnung $p$ legt fest, wie weit das Modell zurückblickt: `AR(1)` nutzt nur den jüngsten Wert, `AR(3)` die letzten drei Werte. Größeres $p$ liefert mehr Kontext, aber auch mehr Gewichte, die später aus Daten gelernt werden müssten. Ein ARMA-Modell erweitert diese Idee zusätzlich um einen Moving-Average-Anteil, der vergangene **Prognosefehler** berücksichtigt.